# Labwork 6 — Regularization, sparsity, and robustness

**Week 2 · Day 6 · ≈ 170 min at the keyboard**

Read Lecture 6 first. This labwork fills in part of the `optlab` package you cloned;
you edit the real source files, and the notebook checks your work as you go.

**What you build today:** proximal methods and lasso, robust regression, the benchmark, and the challenge that tests the whole week's architecture

**Files you will open:**

- `regularizers.py`
- `optimizers/proximal.py`
- `losses.py` (`Huber`, `PoissonNLL`)
- `benchmarks/run.py`, `DESIGN.md`

> **The rule.** `src/optlab/interfaces.py`, `results.py`, `errors.py` and `types.py` are
> **provided** — never edit them. Everything else under `src/optlab/` is yours: replace
> each `raise NotImplementedError` with working code, keeping the signature and honouring
> the docstring.

## Setup

Run this once. It points the notebook at your `optlab` clone and gives you a
`check()` helper that runs a specific test file and reports what happened.

In [ ]:
import subprocess
import sys
from pathlib import Path

# Adjust if your clone lives elsewhere.
OPTLAB = (Path.cwd() / ".." / ".." / "optlab").resolve()
assert OPTLAB.exists(), f"optlab not found at {OPTLAB} -- edit OPTLAB above"
print("optlab:", OPTLAB)


def check(*pytest_args):
    """Run pytest inside the optlab clone and show a short report."""
    r = subprocess.run([sys.executable, "-m", "pytest", "-q", "--no-header", *pytest_args],
                       cwd=OPTLAB, capture_output=True, text=True)
    out = r.stdout + r.stderr
    tail = [l for l in out.splitlines() if l.strip()][-12:]
    print("\n".join(tail))
    if "No module named pytest" in out:
        verdict = "pytest is not installed -- run:  pip install -e '.[dev]'"
    elif r.returncode == 0:
        verdict = "PASSED"
    elif r.returncode == 5:
        # Exit code 5 means pytest collected nothing at all. That is NOT a failure of
        # your code: no test in the suite matches what was asked for. Some days have no
        # automated tests yet; judge those exercises by the checks written in the text.
        verdict = "no tests matched -- nothing to run here, this is not a failure"
    else:
        verdict = "not yet -- keep going"
    print()
    print(verdict)


def edit(relpath):
    """Print the absolute path of a source file, so you can open it in the editor."""
    print(OPTLAB / "src" / "optlab" / relpath)


check("tests/test_no_oracle_in_src.py")   # provided, and already green

---

## Exercise 1 — Proximal gradient and lasso  *(≈ 65 min)*

Implement `L1` and `ElasticNet`, then `ProximalGradient.minimize`.

`L1.prox` is the soft-threshold `sign(v)·max(|v| − λt, 0)`. `L1.gradient` must **raise** —
that is the contract, not a gap. The whole reason `ProximalGradient` asks only for `prox`
is so an honest refusal costs nothing.

The ISTA step is `w⁺ = prox(w − α∇f(w), α)` with `α ≤ 1/L`. Stop on the proximal-gradient
residual `‖w⁺ − w‖/α`, **not** a gradient norm — the gradient of the full objective does not
exist at the solution, which is the entire point.

Then:

1. Verify the hand result: `soft((3, −0.4, 0.1), 0.5) = (2.5, 0, 0)`.
2. Recover a known sparse `w*` and confirm the zeros are **exactly** zero. Compare against
   ridge on the same data — ridge will give you small coefficients and not one true zero.
3. Compute the **regularization path**: sparsity against `λ`. The order in which
   coefficients switch off is a feature-importance ranking the optimizer produced for you.
4. Check against `sklearn.linear_model.Lasso` (oracle, tests only).

`ProximalGradient(GLMLoss(X, y, SquaredError()), L1(lam))` — note `GLMLoss` is day 1's
class, unchanged. A new estimator, and not one existing file edited.

**FISTA is optional.** If you add it, test on an ill-conditioned design: on well-conditioned
data plain ISTA already finishes before acceleration can show any benefit.

**Open:** `src/optlab/regularizers.py`, `src/optlab/optimizers/proximal.py`

In [ ]:
check("tests/unit/test_day6_prox.py", "tests/contracts/test_regularizer_contract.py")

<details>
<summary><b>Plan B</b> — open only if you are stuck for more than ten minutes</summary>

`np.sign(v) * np.maximum(np.abs(v) - t * self.lam, 0.0)`. ElasticNet's prox composes the two: soft-threshold first, then shrink.

</details>

---

## Exercise 2 — Huber and Poisson  *(≈ 40 min)*

Implement `Huber` and `PoissonNLL` as `PointwiseLoss`.

Then confirm the claim that has been building all week: **gradient descent, Newton and
ridge work on them immediately**, with nothing else changed. Fit a robust regression and a
Poisson regression using optimizers you wrote on days 2 and 4.

Three checks: as `delta → ∞`, Huber must reduce to `SquaredError`; a single gross outlier
visibly drags the least-squares line while the Huber fit barely moves; and Poisson agrees
with `sklearn.linear_model.PoissonRegressor`.

Keep the two non-smoothness stories straight. **Huber** is non-smooth nowhere — it is
`C¹`, and it bounds *influence* by capping the derivative at `±δ`. **L1** is non-smooth at
zero in the *parameters*, and that kink is what creates sparsity. Different problems,
different fixes, and the week's last trap is confusing them.

**Open:** `src/optlab/losses.py`

In [ ]:
check("-m", "day6")

---

## Exercise 3 — Benchmark  *(≈ 40 min)*

Fill in `benchmarks/run.py`: every optimizer against every problem, out to a CSV.

Record iterations or epochs, function and gradient evaluations, wall-clock, final loss, and
the gap to the oracle's minimizer. Compare by **epoch** where the method is stochastic —
equal data cost, not equal iteration count.

The deliverable is not the CSV. It is a **one-page** answer to *"which optimizer for which
problem?"*, and it is graded. The honest version names the cases where the sophisticated
method loses: Newton is superb on a small well-conditioned problem and unaffordable when
`p` is large; Adam is unbeatable on badly scaled data and beaten by plain gradient descent
on a clean quadratic.

**Open:** `optlab/benchmarks/run.py`

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "benchmarks/run.py"], cwd=OPTLAB,
                   capture_output=True, text=True)
print((r.stdout + r.stderr).strip()[-1500:])

---

## Exercise 4 — The extensibility challenge  *(≈ 35 min)*

The real exam for the week's architecture.

Add **one** new implementation of an existing interface, **editing no existing file under
`src/`**. Pick one:

- `FISTA` — accelerated `ProximalGradient`
- `GroupLasso` — a `Regularizer` with a block-wise prox
- `Nesterov` — a `DirectionRule`
- `PoissonNLL`, if you did not get to it in Exercise 2

Then run `git diff --stat src/`. If it lists anything other than your new file, you were
forced to edit — and that is the interesting outcome. Record in `DESIGN.md` exactly which
interface was too narrow and what you would change about it.

**An honest account of a design failure is worth more than a lucky success.**

**Open:** a new file under `src/optlab/`, plus `DESIGN.md`

In [ ]:
import subprocess
print(subprocess.run(["git", "diff", "--stat", "src/"], cwd=OPTLAB,
                     capture_output=True, text=True).stdout or "no changes under src/ yet")
print(subprocess.run(["git", "status", "--short", "src/"], cwd=OPTLAB,
                     capture_output=True, text=True).stdout)

---

## Checkpoint

Everything from day 1 to day 6 should be green before you leave, and
`mypy --strict` must be clean. A red type check counts as a failure.

In [ ]:
check("-m", "day1 or day2 or day3 or day4 or day5 or day6")

In [ ]:
r = subprocess.run([sys.executable, "-m", "mypy"], cwd=OPTLAB,
                   capture_output=True, text=True)
print(r.stdout.strip() or r.stderr.strip())

---

## Before the debrief

Be able to answer:

1. Why does L1 create sparsity when ridge does not? Answer in terms of the derivative as
   `w → 0`.
2. What is a proximal operator, and why does having one let you skip the gradient?
3. Why is Huber robust yet smooth — and how does its non-smoothness story differ from L1's?

### Cross code review (20 min)

Swap repositories with another pair and work through this list on **their** code:

- **SRP** — does each class have one reason to change?
- **ISP** — is anything forced to implement a method it does not need?
- **LSP** — do the contract tests pass over *every* implementation?
- **OCP / DIP** — are collaborators injected, or constructed inside?
- **DRY** — is any formula written twice?
- tests, `mypy --strict`, error messages, dead code.

### And that is the week

You have written, from scratch, the optimizers behind linear and logistic regression,
neural-network training, curve fitting, and sparse model selection.

More usefully: for any new problem you can now ask *is it smooth? convex? a finite sum? a
sum of squares? is the penalty differentiable?* — and each answer names the method.